In [1]:
from pyscf import gto, fci, mcscf
import numpy as np
from gpu4pyscf import scf


/home/mert/.local/lib/python3.10/site-packages/gpu4pyscf/lib/cutensor.py:154: UserWarning: using cupy as the tensor contraction engine.
  warnings.warn(f'using {contract_engine} as the tensor contraction engine.')


In [2]:
# Molecular Information part
# !!!!! in CI part mo not forget the change electrons 
# !!!!! also orbitals are important  

xyz = '''
C  -1.941435    2.065642    0.000000
C  -0.620752    2.173981   -0.000000
C  -0.174014    3.618537    0.000000
C  -1.202972    4.453516   -0.000000
B  -2.502193    3.547183   -0.000000
H  -3.636005    3.897822   -0.000000
H  -2.458718    1.126673    0.000000
H   0.085693    1.363887    0.000000
H   0.866414    3.888400    0.000000
H  -1.099878    5.520575   -0.000000'''
molecule = 'C4H5B'
spin = 0
charge =  0


In [4]:
def rhf_calculation(xyz, molecule, spin, charge):
    mol = gto.M(
          atom= xyz,
          basis = 'sto-6g',
          charge = charge,
          verbose = 5 ,
          spin = spin, 
          symmetry = True,
          output  ="./"+molecule+".txt")

    mf = mol.RHF().run()
    mf.analyze() 
    S_ij = mf.get_ovlp()
    C_ij = mf.mo_coeff
    np.savetxt("C_"+molecule+'mo_coeff.txt', C_ij,fmt = '%.8f')
    np.savetxt('S_'+molecule+'_overlap.txt', S_ij,fmt = '%.8f')

    return mf, mol 

In [5]:
def orb_occ_table(orb_energy,orb_occ):
    Homo = None
    Lumo = None
    print ("orbital  energy   occ")

    for i in range (0,(len(orb_occ))):
        print(i ,"   ", orb_energy[i],"  ",orb_occ[i])
              
    for i in range(len(orb_occ) - 1):
        if orb_occ[i] == 2  and orb_occ[i+1] == 0:
            homo = i
            lumo = i + 1
            print(f"Homo_index: {homo}, Lumo_index: {lumo}")
    return homo, lumo


In [6]:
mf, mol  = rhf_calculation( xyz , molecule, spin, charge )
homo , lumo =orb_occ_table(mf.mo_energy, mf.mo_occ)

output file: ./C4H5B.txt
orbital  energy   occ
0     -11.182691079408729    2.0
1     -11.182586522749329    2.0
2     -11.12706284277117    2.0
3     -11.127052714346215    2.0
4     -7.514676973313685    2.0
5     -1.059384210792542    2.0
6     -0.9183578800639551    2.0
7     -0.7827138183134428    2.0
8     -0.6636308733772698    2.0
9     -0.6517932611731614    2.0
10     -0.5939938196962142    2.0
11     -0.5169195179226533    2.0
12     -0.46013485463895476    2.0
13     -0.41625749001807755    2.0
14     -0.40698930074417367    2.0
15     -0.4052510536479954    2.0
16     -0.26286361788988044    2.0
17     0.1535054349401178    0.0
18     0.40215730879155986    0.0
19     0.40863745254049677    0.0
20     0.5990995382026847    0.0
21     0.6082699691910876    0.0
22     0.6584632492959083    0.0
23     0.6825240568760758    0.0
24     0.7430138720838451    0.0
25     0.754119755889819    0.0
26     0.8562339176167866    0.0
27     0.9318554631203684    0.0
28     1.07277534510

In [7]:
n_elec= 4
n_orb = 5 # number of orbital can be determined based on the number of pi  orbitals 
casscf = mcscf.CASSCF(mf, n_orb, n_elec)
'''output in order
e_tot
e_cas
Fci vecs 
mo_coeff
mo_energy'''
mo = casscf.sort_mo([15,17,18,19,20])# the numbers are from the index of pi orbitals 
casscf.kernel(mo) 


(np.float64(-178.65014896429267),
 np.float64(-3.3691278451967435),
 FCIvector([[-9.35677909e-01,  0.00000000e+00, -4.52770728e-03,
             -1.73522950e-02,  0.00000000e+00, -2.23448034e-02,
              0.00000000e+00, -3.19793198e-03,  0.00000000e+00,
             -8.10729705e-03],
            [ 0.00000000e+00,  1.36777186e-01,  0.00000000e+00,
              0.00000000e+00,  8.76204569e-02,  0.00000000e+00,
             -8.60936367e-02,  0.00000000e+00,  7.56533773e-04,
              0.00000000e+00],
            [-4.52770728e-03,  0.00000000e+00,  9.00112656e-02,
              1.09964139e-01,  0.00000000e+00, -7.47271967e-03,
              0.00000000e+00, -3.84805950e-02,  0.00000000e+00,
             -2.26905922e-03],
            [-1.73522950e-02,  0.00000000e+00,  1.09964139e-01,
              1.01137824e-01,  0.00000000e+00,  4.13918554e-03,
              0.00000000e+00, -6.13934046e-02,  0.00000000e+00,
              3.98676306e-03],
            [ 0.00000000e+00,  8.7620456

In [8]:
casscf.fcisolver.max_cycle = 3000
casscf.canonicalize(mo_coeff=None, ci=None, eris=None, sort=False, cas_natorb=False, casdm1=None, verbose=3, with_meta_lowdin=False);

filename = f"{molecule}_Ci_coeffs.txt"

with open(filename, 'w', encoding='utf-8') as f:
    
    f.write("det-alpha,     det-beta,    CI coefficients\n")
    
    
    for c, ia, ib in casscf.fcisolver.large_ci(casscf.ci, n_orb, n_elec, tol=.000000001, return_strs=False):
        # Satır formatını oluşturalım
        line = f" {str(ia):<12} {str(ib):<12} {c:.15f}\n"
        
        #write in file 
        f.write(line)
        #show in terminal 
        print(line.strip())

[0 1]        [0 1]        -0.935677908815278
[0 1]        [1 2]        -0.004527707276507
[0 1]        [0 3]        -0.017352295022069
[0 1]        [2 3]        -0.022344803351778
[0 1]        [1 4]        -0.003197931976407
[0 1]        [3 4]        -0.008107297046536
[0 2]        [0 2]        0.136777185945000
[0 2]        [1 3]        0.087620456932747
[0 2]        [0 4]        -0.086093636722583
[0 2]        [2 4]        0.000756533773031
[1 2]        [0 1]        -0.004527707276507
[1 2]        [1 2]        0.090011265617996
[1 2]        [0 3]        0.109964139277719
[1 2]        [2 3]        -0.007472719671471
[1 2]        [1 4]        -0.038480595005359
[1 2]        [3 4]        -0.002269059217625
[0 3]        [0 1]        -0.017352295022069
[0 3]        [1 2]        0.109964139277719
[0 3]        [0 3]        0.101137824157160
[0 3]        [2 3]        0.004139185536441
[0 3]        [1 4]        -0.061393404568900
[0 3]        [3 4]        0.003986763056846
[1 3]        [0 2] 